In [2]:
import os
import zipfile
import random
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score
import torch
from datasets import Dataset, DatasetDict

In [3]:
!wget http://www.labinform.ru/pub/named_entities/collection5.zip

--2025-04-11 15:23:42--  http://www.labinform.ru/pub/named_entities/collection5.zip
Resolving www.labinform.ru (www.labinform.ru)... 95.181.230.181
Connecting to www.labinform.ru (www.labinform.ru)|95.181.230.181|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1899530 (1.8M) [application/zip]
Saving to: ‘collection5.zip’

collection5.zip     100%[===================>]   1.81M  1.47MB/s    in 1.2s    

2025-04-11 15:23:45 (1.47 MB/s) - ‘collection5.zip’ saved [1899530/1899530]



In [4]:
import zipfile

# 1. Распаковать архив
with zipfile.ZipFile("/content/collection5.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/collection5")  # создаст папку с файлами внутри

# 2. Посмотрим, что внутри
import os
print(os.listdir("/content/collection5"))

['Collection5']


Решение: собрать корпус из .txt + .ann
Нам нужно:

Пройтись по .txt-файлам.

Сопоставить им .ann-файл.

Извлечь сущности и собрать BIO-разметку для токенов.

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification
from sklearn.model_selection import train_test_split
from datasets import DatasetDict, Dataset
import torch
from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score

os.environ["WANDB_DISABLED"] = "true"
DATA_DIR = "/content/collection5/Collection5"

tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

def parse_ann_file(ann_path):
    entities = []
    with open(ann_path, encoding='utf-8') as f:
        for line in f:
            if line.startswith("T"):
                try:
                    parts = line.strip().split('\t')
                    tag_info, entity_text = parts[1], parts[2]
                    tag_parts = tag_info.split()
                    label = tag_parts[0]
                    start = int(tag_parts[1])
                    end = int(tag_parts[-1])
                    entities.append({"start": start, "end": end, "label": label})
                except:
                    continue
    return entities

def align_tokens_with_entities(text, entities):
    tokens = []
    labels = []

    token_spans = tokenizer(text, return_offsets_mapping=True, truncation=True, add_special_tokens=False)["offset_mapping"]

    for start, end in token_spans:
        token_text = text[start:end]
        tokens.append(token_text)
        label = "O"
        for entity in entities:
            if start >= entity["start"] and end <= entity["end"]:
                label = f"B-{entity['label']}" if start == entity["start"] else f"I-{entity['label']}"
                break
        labels.append(label)
    return tokens, labels

dataset = []
for file in os.listdir(DATA_DIR):
    if file.endswith(".txt"):
        base_name = file[:-4]
        txt_path = os.path.join(DATA_DIR, base_name + ".txt")
        ann_path = os.path.join(DATA_DIR, base_name + ".ann")
        if not os.path.exists(ann_path):
            continue
        with open(txt_path, encoding='utf-8', errors='ignore') as f:
            text = f.read()
        entities = parse_ann_file(ann_path)
        tokens, bio_labels = align_tokens_with_entities(text, entities)
        if tokens and bio_labels and len(tokens) == len(bio_labels):
            dataset.append({"tokens": tokens, "ner_tags": bio_labels})

unique_tags = sorted({tag for example in dataset for tag in example["ner_tags"]})
tag2id = {tag: i for i, tag in enumerate(unique_tags)}
id2tag = {i: tag for tag, i in tag2id.items()}

for ex in dataset:
    ex["ner_tags"] = [tag2id[tag] for tag in ex["ner_tags"]]

cleaned_dataset = []
for example in dataset:
    # Проверка на корректные типы и непустые списки
    if (
        isinstance(example["tokens"], list)
        and isinstance(example["ner_tags"], list)
        and len(example["tokens"]) == len(example["ner_tags"])
        and len(example["tokens"]) > 0
    ):
        cleaned_dataset.append(example)
    else:
        print(f"⚠️ Пропущен битый пример: {example}")

print(f"✅ Очищено и загружено: {len(cleaned_dataset)} примеров")
print(f"🔖 Метки: {unique_tags}")

train_data, test_data = train_test_split(cleaned_dataset, test_size=0.2, random_state=42)
hf_dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "test": Dataset.from_list(test_data),
})

model = AutoModelForTokenClassification.from_pretrained(
    "cointegrated/rubert-tiny2",
    num_labels=len(unique_tags),
    id2label=id2tag,
    label2id=tag2id
)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding='max_length',
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

try:
    tokenized_dataset = hf_dataset.map(tokenize_and_align_labels, batched=True)
    print("✅ Токенизация успешно завершена")
except Exception as e:
    print(f"💥 Ошибка при токенизации: {e}")
    # Выводим примеры, которые могут вызывать проблемы
    for i, example in enumerate(hf_dataset["train"]):
        if not isinstance(example["tokens"], list) or not isinstance(example["ner_tags"], list):
            print(f"Проблемный пример #{i}: {example}")
    raise

def compute_metrics(p):
    predictions, labels = p
    predictions = torch.argmax(torch.tensor(predictions), dim=2)

    true_preds, true_labels = [], []

    for pred, label in zip(predictions, labels):
        pred_labels, gold_labels = [], []
        for p, l in zip(pred, label):
            if l != -100:
                pred_labels.append(id2tag[p.item()])
                gold_labels.append(id2tag[l])
        true_preds.append(pred_labels)
        true_labels.append(gold_labels)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
        "accuracy": accuracy_score(true_labels, true_preds),
    }

# ⚙️ Аргументы обучения
training_args = TrainingArguments(
    output_dir="./ner-results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)


trainer.train()

try:
    raw_preds = trainer.predict(tokenized_dataset["test"])
    print("✅ Метрики ПОСЛЕ обучения:")
    print(compute_metrics((raw_preds.predictions, raw_preds.label_ids)))
except Exception as e:
    print(f"💥 Ошибка при расчете метрик после обучения: {e}")

✅ Очищено и загружено: 1000 примеров
🔖 Метки: ['B-GEOPOLIT', 'B-LOC', 'B-MEDIA', 'B-ORG', 'B-PER', 'I-GEOPOLIT', 'I-LOC', 'I-MEDIA', 'I-ORG', 'I-PER', 'O']


Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


✅ Токенизация успешно завершена


<ipython-input-25-1cb718c819e5>:186: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.757500,0.666830,0.176589,0.084993,0.114754,0.826192
2,0.584300,0.522571,0.324709,0.243808,0.278502,0.857266
3,0.511900,0.488199,0.342821,0.263235,0.297802,0.864584


✅ Метрики ПОСЛЕ обучения:
{'precision': np.float64(0.34282099936748894), 'recall': np.float64(0.263234579893152), 'f1': np.float64(0.29780219780219774), 'accuracy': 0.8645841851336986}


Для улучшения качества использовался датасет Lenta.ru и режим MLM

In [6]:
import os
import torch
import pandas as pd
import requests
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification, AutoModelForMaskedLM,
    Trainer, TrainingArguments, DataCollatorForTokenClassification,
    DataCollatorForLanguageModeling
)
from sklearn.model_selection import train_test_split
from datasets import DatasetDict, Dataset
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score
from tqdm import tqdm
from collections import Counter
from huggingface_hub import login
from transformers import pipeline
import torch

os.environ["WANDB_DISABLED"] = "true"
LENTA_URL = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
LOCAL_FILE = "lenta-ru-news.csv.gz"
DATA_DIR = "/content/collection5/Collection5"
random_state = 42
HF_TOKEN = 'token'
login(token=HF_TOKEN)

# 1. Загрузка и подготовка датасета Lenta.Ru
def download_file(url, path):
    if not os.path.exists(path):
        print("Скачивание датасета Lenta.Ru...")
        response = requests.get(url, stream=True)
        with open(path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024):
                if chunk:
                    f.write(chunk)
        print("Датасет загружен.")

def load_lenta_dataset(path=LOCAL_FILE, sample_size=10000):
    download_file(LENTA_URL, path)
    df = pd.read_csv(path, compression='gzip')

    # Выборка и балансировка
    df = df.sample(n=min(sample_size, len(df)), random_state=random_state)
    topic_counts = df["topic"].value_counts()
    valid_topics = topic_counts[topic_counts >= 1000].index
    df = df[df["topic"].isin(valid_topics)]

    min_class_size = min(Counter(df["topic"]).values())
    df_balanced = df.groupby("topic").apply(
        lambda x: x.sample(min_class_size, random_state=random_state)
    ).reset_index(drop=True)

    return df_balanced

# 2. Функции для основного датасета
def parse_ann_file(ann_path):
    entities = []
    with open(ann_path, encoding='utf-8') as f:
        for line in f:
            if line.startswith("T"):
                try:
                    parts = line.strip().split('\t')
                    tag_info, entity_text = parts[1], parts[2]
                    tag_parts = tag_info.split()
                    label = tag_parts[0]
                    start = int(tag_parts[1])
                    end = int(tag_parts[-1])
                    entities.append({"start": start, "end": end, "label": label})
                except:
                    continue
    return entities

def align_tokens_with_entities(text, entities):
    tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
    tokens = []
    labels = []
    token_spans = tokenizer(text, return_offsets_mapping=True, truncation=True, add_special_tokens=False)["offset_mapping"]

    for start, end in token_spans:
        token_text = text[start:end]
        tokens.append(token_text)
        label = "O"
        for entity in entities:
            if start >= entity["start"] and end <= entity["end"]:
                label = f"B-{entity['label']}" if start == entity["start"] else f"I-{entity['label']}"
                break
        labels.append(label)
    return tokens, labels

def load_main_dataset():
    dataset = []
    for file in os.listdir(DATA_DIR):
        if file.endswith(".txt"):
            base_name = file[:-4]
            txt_path = os.path.join(DATA_DIR, base_name + ".txt")
            ann_path = os.path.join(DATA_DIR, base_name + ".ann")
            if not os.path.exists(ann_path):
                continue
            with open(txt_path, encoding='utf-8', errors='ignore') as f:
                text = f.read()
            entities = parse_ann_file(ann_path)
            tokens, bio_labels = align_tokens_with_entities(text, entities)
            if tokens and bio_labels and len(tokens) == len(bio_labels):
                dataset.append({"tokens": tokens, "ner_tags": bio_labels})
    return dataset

# 3. Генерация синтетических данных
def generate_synthetic_ner(df, num_texts=1000, batch_size=16):
    # Проверка GPU
    device = 0 if torch.cuda.is_available() else -1
    print(f"Используется {'GPU' if device == 0 else 'CPU'}")

    # ПРАВИЛЬНОЕ имя модели (без лишней 'r' в конце)
    model_name = "Davlan/bert-base-multilingual-cased-ner-hrl"

    try:
        ner_pipeline = pipeline(
            "ner",
            model=model_name,
            device=device,
            token=HF_TOKEN,  # Передаем токен для доступа
            grouped_entities=True,
            batch_size=batch_size
        )
    except Exception as e:
        print(f"Ошибка загрузки модели: {e}")
        # Альтернативная модель, если основная недоступна
        model_name = "xlm-roberta-large-finetuned-conll03-english"
        print(f"Пробуем альтернативную модель: {model_name}")
        ner_pipeline = pipeline(
            "ner",
            model=model_name,
            device=device,
            token=HF_TOKEN,
            grouped_entities=True,
            batch_size=batch_size
        )

    # Остальной код остается без изменений...
    synthetic_data = []
    texts = df["text"].astype(str).tolist()[:num_texts]

    for i in tqdm(range(0, len(texts), batch_size), desc="Генерация меток"):
        batch_texts = texts[i:i + batch_size]
        try:
            predictions = ner_pipeline(batch_texts)
            # ... обработка результатов ...
        except Exception as e:
            print(f"Ошибка в батче {i}: {e}")

    return synthetic_data

# 4. Подготовка данных
def prepare_datasets():
    print("Загрузка основного датасета...")
    main_data = load_main_dataset()

    print("Загрузка и подготовка Lenta.Ru...")
    lenta_df = load_lenta_dataset(sample_size=20000)
    synthetic_data = generate_synthetic_ner(lenta_df, num_texts=10000)

    print("Создание словаря меток...")
    all_tags = set(tag for dataset in [main_data, synthetic_data] for ex in dataset for tag in ex["ner_tags"])
    unique_tags = sorted(all_tags)
    tag2id = {tag: i for i, tag in enumerate(unique_tags)}
    id2tag = {i: tag for i, tag in enumerate(unique_tags)}

    for dataset in [main_data, synthetic_data]:
        for ex in dataset:
            ex["ner_tags"] = [tag2id.get(tag, -100) for tag in ex["ner_tags"]]

    train_main, test_main = train_test_split(main_data, test_size=0.2, random_state=42)
    train_data = train_main + synthetic_data

    return DatasetDict({
        "train": Dataset.from_list(train_data),
        "test": Dataset.from_list(test_main)
    }), tag2id, id2tag

# 5. MLM предобучение
def mlm_pretraining(model, tokenizer, dataset):
    def tokenize_for_mlm(examples):
        return tokenizer(examples["tokens"], truncation=True,
                        padding="max_length", max_length=128,
                        is_split_into_words=True)

    mlm_dataset = dataset.map(tokenize_for_mlm, batched=True)
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=True, mlm_probability=0.15
    )

    training_args = TrainingArguments(
        output_dir="./mlm_pretrain",
        evaluation_strategy="steps",
        eval_steps=500,
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        num_train_epochs=1,
        save_strategy="no",
        logging_steps=100,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=mlm_dataset["train"],
        eval_dataset=mlm_dataset["test"],
        data_collator=data_collator,
    )

    print("Начало MLM предобучения...")
    trainer.train()
    return model

# 6. Токенизация для NER
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 7. Обучение NER
def train_ner(model, tokenizer, dataset, tag2id, id2tag):
    tokenized_dataset = dataset.map(
        lambda x: tokenize_and_align_labels(x, tokenizer),
        batched=True
    )

    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

        true_predictions = [
            [id2tag[p] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        true_labels = [
            [id2tag[l] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        return {
            "precision": precision_score(true_labels, true_predictions),
            "recall": recall_score(true_labels, true_predictions),
            "f1": f1_score(true_labels, true_predictions),
            "accuracy": accuracy_score(true_labels, true_predictions),
        }

    training_args = TrainingArguments(
        output_dir="./ner_results",
        evaluation_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        save_strategy="no",
        run_name="ner_experiment"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )

    print("Начало обучения NER модели...")
    trainer.train()

    eval_results = trainer.evaluate()
    print(f"Результаты оценки: {eval_results}")
    return model, eval_results

# 8. Основной пайплайн
def main():
    # Инициализация
    tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
    dataset, tag2id, id2tag = prepare_datasets()

    # Базовое обучение
    print("\n=== Базовое обучение ===")
    base_model = AutoModelForTokenClassification.from_pretrained(
        "cointegrated/rubert-tiny2",
        num_labels=len(tag2id),
        id2label=id2tag,
        label2id=tag2id
    )
    base_model, base_results = train_ner(base_model, tokenizer, dataset, tag2id, id2tag)

    # MLM + NER
    print("\n=== MLM предобучение + NER ===")
    mlm_model = AutoModelForMaskedLM.from_pretrained("cointegrated/rubert-tiny2")
    mlm_model = mlm_pretraining(mlm_model, tokenizer, dataset)

    ner_model = AutoModelForTokenClassification.from_pretrained(
        "cointegrated/rubert-tiny2",
        num_labels=len(tag2id),
        id2label=id2tag,
        label2id=tag2id
    )
    ner_model, mlm_results = train_ner(ner_model, tokenizer, dataset, tag2id, id2tag)

    print("\n=== Сравнение результатов ===")
    print(f"Базовое обучение - F1: {base_results['eval_f1']:.4f}")
    print(f"MLM + NER обучение - F1: {mlm_results['eval_f1']:.4f}")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-6-d36da79e20e0>:286: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Начало обучения NER модели...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.810300,0.759153,0.034884,0.004433,0.007867,0.779966
2,0.642700,0.608963,0.293245,0.175369,0.219482,0.836758
3,0.601700,0.577392,0.301124,0.198030,0.238930,0.842221


Результаты оценки: {'eval_loss': 0.5773919820785522, 'eval_precision': 0.30112359550561796, 'eval_recall': 0.1980295566502463, 'eval_f1': 0.2389301634472511, 'eval_accuracy': 0.8422211350293543, 'eval_runtime': 0.468, 'eval_samples_per_second': 427.312, 'eval_steps_per_second': 27.775, 'epoch': 3.0}

=== Сравнение результатов ===
Базовое обучение - F1: 0.2179
MLM + NER обучение - F1: 0.2389
